In [ ]:
from rapidfuzz import fuzz, process
import pandas as pd

# --- Configuration ---
SIMILARITY_THRESHOLD = 85  # Titles with similarity >= this are considered duplicates
df = pd.read_csv("pubmed_r2_title.csv")

# --- Step 1: Normalize titles for comparison ---
def normalize_title(title):
    """Lowercase and strip punctuation/extra whitespace for cleaner comparison."""
    import re
    if pd.isna(title):
        return ""
    title = str(title).lower().strip()
    title = re.sub(r'[^\w\s]', '', title)   # remove punctuation
    title = re.sub(r'\s+', ' ', title)       # collapse whitespace
    return title

df['_normalized_title'] = df['Title'].apply(normalize_title)

# --- Step 2: Find all duplicate/near-duplicate groups ---
titles = df['_normalized_title'].tolist()
raw_titles = df['Title'].tolist()  # keep raw titles for prefix check
visited = set()
duplicate_groups = []

for i, title_i in enumerate(titles):
    if i in visited:
        continue
    group = [i]
    for j, title_j in enumerate(titles):
        if i == j or j in visited:
            continue
        # Skip if either title is an "Author's Reply:" entry
        if "reply:" in str(raw_titles[i]).lower() or "reply:" in str(raw_titles[j]).lower():
            continue
        score = fuzz.token_sort_ratio(title_i, title_j)
        if score >= SIMILARITY_THRESHOLD:
            group.append(j)
    if len(group) > 1:
        visited.update(group)
        duplicate_groups.append(group)

# --- Step 3: Print all duplicate/near-duplicate groups ---
print("=" * 60)
print("DUPLICATE / NEAR-DUPLICATE GROUPS FOUND:")
print("=" * 60)

if not duplicate_groups:
    print("No duplicates found.")
else:
    for group_num, group in enumerate(duplicate_groups, 1):
        print(f"\nGroup {group_num}:")
        for idx in group:
            print(f"[Row {idx}] {df['Title'].iloc[idx]!r}")

# --- Step 4: Deduplicate — keep first occurrence in each group ---
rows_to_drop = set()
for group in duplicate_groups:
    rows_to_drop.update(group[1:])  # keep group[0], drop the rest

df_deduped = df.drop(index=df.index[list(rows_to_drop)]).drop(columns=['_normalized_title']).reset_index(drop=True)

# Also clean up the helper column on the original df
df.drop(columns=['_normalized_title'], inplace=True)

# --- Step 5: Print final length ---
print("\n" + "=" * 60)
print(f"Original df length:    {len(df)}")
print(f"Deduplicated df length: {len(df_deduped)}")
print(f"Rows removed:          {len(df) - len(df_deduped)}")
print("=" * 60)